# GA 2d

Reusable function definitions only. Original project inputs, run settings and saved outputs are not included. Running these cells defines functions; it does not read a dataset. Start with the [synthetic walkthrough](../examples/synthetic_walkthrough.ipynb) for a complete small run.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "helper" / "pipeline_utils.py").is_file()
             and (p / "examples" / "synthetic_demo.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start the notebook inside this repository.")
for folder in (ROOT, ROOT / "Invert-PropOnly-saveCSV", ROOT / "Invert-Prop+cond-saveCSV"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))


In [ ]:
import numpy as np
import random
from typing import List, Tuple

from invert_two_utils import (
    initialize_two_output_inversion,
    fitness_two_outputs,
    unpack_two_output_solutions,
    select_top_k_unique
)

def invert_two_outputs_ga_scaled_Q(
    x_scaler_path: str,
    y_scaler_path: str,
    model1_path: str,
    model2_path: str,
    x_fixed_raw: np.ndarray,        # raw fixed inputs (len 10)
    actual_x_adj_raw: np.ndarray,   # raw adjustable inputs (len 4)
    target1_raw: float,
    target2_raw: float,
    pop_size: int,
    num_generations: int,
    crossover_rate: float,
    mutation_rate: float,
    elitism_count: int,
    hidden_dims1: List[int],
    hidden_dims2: List[int],
    activation: str,
    x_adj_intervals_raw: np.ndarray,  # raw [min,max] for each adjustable input (4×2)
    Q_bounds: Tuple[float, float],     # (Q_min, Q_max) in raw space
    k: int                             # number of top solutions to return
) -> List[Tuple[np.ndarray, np.ndarray, np.ndarray]]:
    """
    Genetic‐Algorithm inversion for two network outputs under
    x1_raw = Q * x0_raw constraint.
    
    Returns up to k+1 tuples:
      (adj_inputs_raw (4,), preds_raw (2,), resid_raw (2,))
    with the first entry always the “actual” test‐data point.
    """

    # [1] Shared initialization: scalers, models, scaled inputs, actual result, bounds, scaled targets
    init = initialize_two_output_inversion(
        x_scaler_path, y_scaler_path,
        model1_path, model2_path,
        hidden_dims1, hidden_dims2,
        activation,
        x_fixed_raw, actual_x_adj_raw,
        target1_raw, target2_raw,
        x_adj_intervals_raw, Q_bounds
    )
    # Unpack shared data
    model1, model2 = init["model1"], init["model2"]
    means_x, scales_x = init["means_x"], init["scales_x"]
    scales_y, mean_y = init["scales_y"], init["mean_y"]
    x_fixed_s = init["x_fixed_s"]
    results = init["results"]         # starts with actual test‐data result
    min_vec, max_vec = init["min_vec"], init["max_vec"]
    target_s = init["target_s"]

    # [2] Initialize GA population within the search bounds
    population = [np.random.uniform(min_vec, max_vec) for _ in range(pop_size)]

    # [3] Run GA: selection, crossover, mutation, elitism
    for _ in range(num_generations):
        # Evaluate fitness for each individual
        scored = [
            (fitness_two_outputs(ind, model1, model2, x_fixed_s, means_x, scales_x, target_s), ind)
            for ind in population
        ]
        scored.sort(key=lambda t: t[0])

        # Elitism: carry forward top performers
        new_pop = [ind for (_, ind) in scored[:elitism_count]]

        # Fill the rest via tournament selection, crossover, and mutation
        while len(new_pop) < pop_size:
            # 3-way tournament selection
            p1 = min(random.sample(scored, 3), key=lambda t: t[0])[1]
            p2 = min(random.sample(scored, 3), key=lambda t: t[0])[1]

            # Uniform crossover
            if random.random() < crossover_rate:
                mask = np.random.rand(4) < 0.5
                child = np.where(mask, p1, p2)
            else:
                child = p1.copy()

            # Mutation: random reset
            for j in range(4):
                if random.random() < mutation_rate:
                    child[j] = np.random.uniform(min_vec[j], max_vec[j])

            new_pop.append(child)

        population = new_pop

    # [4] Select top-k unique scaled candidates by fitness
    final_fitnesses = [
        fitness_two_outputs(ind, model1, model2, x_fixed_s, means_x, scales_x, target_s)
        for ind in population
    ]
    best_scaled = select_top_k_unique(final_fitnesses, population, k)

    # [5] Inverse‐scale candidates → raw inputs, raw predictions & residuals
    solved = unpack_two_output_solutions(
        best_scaled,
        model1, model2,
        means_x, scales_x,
        x_fixed_s,
        scales_y, mean_y,
        target1_raw, target2_raw
    )

    # Return actual test‐data result + GA solutions
    return results + solved
